# RQ4 — margin as a fourth abstention signal

Re-benchmark risk–coverage / AURC on both datasets with **UMLS candidate margin**
(`margin_mean`) beside entropy, mapping-confidence, random, the old 2-signal
lexicographic `combined`, and **combined_3** = mean of rank-normalised
(entropy, confidence, margin).

Higher encoded score = safer. Rank-normalise within (dataset, model) before combining.
Do not impute missing margin. MedMentions encoders excluded (direct-CUI).
OpenBioLLM on CADEC is kept on defined-margin rows only; `n` is flagged.

In [ ]:
# === Setup (reuse RQ4 Clinical Utility AURC helpers) ==========================
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

PROJECT_ROOT = Path.home() / "projects" / "Measuring-Semantic-Stability-in-Clinical-LLMs"
assert (PROJECT_ROOT / "config.json").is_file()

OUT_DIR = PROJECT_ROOT / "outputs" / "rq4"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ALL_FIG = PROJECT_ROOT / "all_rq_figures"
ALL_FIG.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "MedMentions": {
        "label": "MedMentions (biomedical literature)",
        "path": PROJECT_ROOT / "outputs/rq1/umls_candidate_margin_medmentions.csv",
        "entropy_col": "normalised_semantic_entropy_full",
        "accuracy_col": "mean_accuracy_full",
        "confidence_col": "mapping_confidence",
        "exclude_models": ["BERT-base", "BioBERT", "PubMedBERT"],
        "exclude_reason": "direct-CUI; margin undefined",
    },
    "CADEC": {
        "label": "CADEC (patient-generated health text)",
        "path": PROJECT_ROOT / "outputs/rq3/umls_candidate_margin_cadec.csv",
        "entropy_col": "normalised_entropy",
        "accuracy_col": "accuracy",
        "confidence_col": "mapping_confidence",
        "exclude_models": [],
        "exclude_reason": "",
    },
}

MODEL_ORDER = [
    "BERT-base", "BioBERT", "PubMedBERT", "FLAN-T5-base",
    "BioMistral-7B", "Mistral-7B-Instruct-v0.1",
    "Llama3-OpenBioLLM-8B", "Meta-Llama-3-8B-Instruct",
]
SMALL_N_MODEL = "Llama3-OpenBioLLM-8B"
SMALL_N_DATASET = "CADEC"
# Only OpenBioLLM-on-CADEC is small-n (~233). Do NOT flag MedMentions n=491.

# Same grid as RQ4 Clinical Utility: 1.00 .. 0.10 step 0.05
COVERAGE_GRID = np.round(np.arange(1.00, 0.09, -0.05), 2)
N_RANDOM = 20
RNG = np.random.default_rng(42)

for cfg in DATASETS.values():
    assert cfg["path"].is_file(), cfg["path"]
    print(f"OK {cfg['label']}: {cfg['path']}")


In [ ]:
# === AURC helpers copied from RQ4_Clinical_Utility_Abstention (do not change) =
def selective_curve(y, score_order, coverages=COVERAGE_GRID):
    """score_order: indices ranked best-first (keep prefix)."""
    ranked_y = y[score_order]
    n = len(y)
    rows = []
    for cov in coverages:
        k = max(1, int(np.ceil(float(cov) * n)))
        acc = float(np.mean(ranked_y[:k]))
        rows.append({
            "coverage": float(cov),
            "n_keep": int(k),
            "n_total": int(n),
            "selective_accuracy": acc,
            "risk": 1.0 - acc,
        })
    return rows


def aurc_from_curve(coverages, risks):
    c = np.asarray(coverages, dtype=float)
    r = np.asarray(risks, dtype=float)
    order = np.argsort(c)
    trapz = getattr(np, "trapezoid", None) or np.trapz
    return float(trapz(r[order], c[order]))


def order_entropy(h, conf):
    return np.argsort(h, kind="mergesort")


def order_confidence(h, conf):
    return np.argsort(-conf, kind="mergesort")


def order_combined(h, conf):
    """Lexicographic 2-signal: low entropy first; high confidence breaks ties."""
    return np.lexsort((-conf, h))


def percentile_safe(x, higher_is_safer=True):
    """Rank-normalise to (0, 1] within the current model; higher = safer."""
    r = pd.Series(x).rank(method="average", pct=True).to_numpy(dtype=float)
    return r if higher_is_safer else (1.0 - r)


In [ ]:
# === Load: drop NaN margin; MM encoders out; never impute =====================
def load_margin_dataset(ds_key: str) -> pd.DataFrame:
    cfg = DATASETS[ds_key]
    df = pd.read_csv(cfg["path"])
    ecol, acol, ccol = cfg["entropy_col"], cfg["accuracy_col"], cfg["confidence_col"]
    need = ["instance_id", "model_name", ecol, acol, ccol, "margin_mean", "margin_min"]
    miss = [c for c in need if c not in df.columns]
    assert not miss, f"{ds_key} missing {miss}"

    n_raw = len(df)
    if cfg["exclude_models"]:
        n_ex = df["model_name"].isin(cfg["exclude_models"]).sum()
        print(f"{ds_key}: excluding {cfg['exclude_models']} ({n_ex} rows; {cfg['exclude_reason']})")
        df = df[~df["model_name"].isin(cfg["exclude_models"])].copy()

    out = df[need].rename(columns={
        ecol: "entropy", acol: "accuracy", ccol: "confidence",
    }).copy()
    for col in ["entropy", "accuracy", "confidence", "margin_mean", "margin_min"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    before = len(out)
    # Do NOT impute missing margin.
    out = out.dropna(subset=["entropy", "accuracy", "confidence", "margin_mean"]).copy()
    out["entropy"] = out["entropy"].clip(lower=0.0)
    out["dataset"] = ds_key
    out["dataset_label"] = cfg["label"]
    print(
        f"{ds_key}: kept {len(out):,}/{before:,} (raw file {n_raw:,}) | "
        f"models={out.model_name.nunique()} | "
        f"mean_H={out.entropy.mean():.3f} mean_acc={out.accuracy.mean():.3f} "
        f"mean_conf={out.confidence.mean():.3f} mean_margin={out.margin_mean.mean():.3f}"
    )
    counts = out.groupby("model_name").size()
    for m in MODEL_ORDER:
        if m not in counts.index:
            continue
        n = int(counts[m])
        flag = "  [SMALL-n FLAG]" if (ds_key == SMALL_N_DATASET and m == SMALL_N_MODEL) else ""
        print(f"  {m:28s} n={n}{flag}")
    return out


FRAMES = {k: load_margin_dataset(k) for k in DATASETS}


In [ ]:
# === Risk–coverage + AURC + Spearman ==========================================
SIGNALS_ORDER = [
    "entropy", "confidence", "margin", "random", "combined", "combined_3", "margin_min",
]


def _aurc(curve):
    return aurc_from_curve(
        [r["coverage"] for r in curve],
        [r["risk"] for r in curve],
    )


def run_dataset(ds_key: str, df: pd.DataFrame):
    cfg = DATASETS[ds_key]
    curve_rows, aurc_tidy, spear_rows = [], [], []
    models = [m for m in MODEL_ORDER if m in set(df.model_name)]

    for model in models:
        g = df[df["model_name"] == model].reset_index(drop=True)
        y = g["accuracy"].to_numpy(dtype=float)
        h = g["entropy"].to_numpy(dtype=float)
        conf = g["confidence"].to_numpy(dtype=float)
        marg = g["margin_mean"].to_numpy(dtype=float)
        marg_min = g["margin_min"].to_numpy(dtype=float)
        n = len(y)
        small_n = bool(ds_key == SMALL_N_DATASET and model == SMALL_N_MODEL)

        # Rank-normalise WITHIN (dataset, model); higher = safer.
        h_safe = percentile_safe(h, higher_is_safer=False)
        c_safe = percentile_safe(conf, higher_is_safer=True)
        m_safe = percentile_safe(marg, higher_is_safer=True)
        mmin_safe = percentile_safe(marg_min, higher_is_safer=True)
        combined_3 = (h_safe + c_safe + m_safe) / 3.0

        orders = {
            "entropy": order_entropy(h, conf),           # raw; ≡ rank of h_safe desc
            "confidence": order_confidence(h, conf),     # raw; ≡ rank of c_safe desc
            "margin": np.argsort(-marg, kind="mergesort"),
            "margin_min": np.argsort(-marg_min, kind="mergesort"),
            "combined": order_combined(h, conf),         # existing 2-signal lex
            "combined_3": np.argsort(-combined_3, kind="mergesort"),
        }

        curves = {}
        for signal, order in orders.items():
            curve = selective_curve(y, order)
            for r in curve:
                r.update({"dataset": ds_key, "model": model, "signal": signal, "n": n, "small_n": small_n})
                curve_rows.append(r)
            curves[signal] = curve

        # Random = mean of 20 seeded permutations (shared RNG(42))
        acc_by_cov = {float(c): [] for c in COVERAGE_GRID}
        for _ in range(N_RANDOM):
            perm = RNG.permutation(n)
            ranked_y = y[perm]
            for cov in COVERAGE_GRID:
                k = max(1, int(np.ceil(float(cov) * n)))
                acc_by_cov[float(cov)].append(float(np.mean(ranked_y[:k])))
        rand_curve = []
        for cov in COVERAGE_GRID:
            acc = float(np.mean(acc_by_cov[float(cov)]))
            row = {
                "dataset": ds_key, "model": model, "signal": "random",
                "coverage": float(cov),
                "n_keep": max(1, int(np.ceil(float(cov) * n))),
                "n_total": int(n), "n": n, "small_n": small_n,
                "selective_accuracy": acc, "risk": 1.0 - acc,
            }
            curve_rows.append(row)
            rand_curve.append(row)
        curves["random"] = rand_curve

        aurc_map = {sig: _aurc(curves[sig]) for sig in SIGNALS_ORDER}
        singles = [aurc_map["entropy"], aurc_map["confidence"], aurc_map["margin"]]
        best_single = float(np.min(singles))
        best_single_name = ["entropy", "confidence", "margin"][int(np.argmin(singles))]
        c3 = aurc_map["combined_3"]
        c2 = aurc_map["combined"]
        c3_beats_best_single = bool(c3 < best_single - 1e-15)
        c3_beats_combined2 = bool(c3 < c2 - 1e-15)
        c3_beats_both = bool(c3_beats_best_single and c3_beats_combined2)

        for sig in SIGNALS_ORDER:
            aurc_tidy.append({
                "dataset": ds_key,
                "dataset_label": cfg["label"],
                "model": model,
                "n": n,
                "small_n": small_n,
                "signal": sig,
                "AURC": aurc_map[sig],
                "best_single_signal": best_single_name,
                "best_single_AURC": best_single,
                "combined_3_lt_best_single": c3_beats_best_single,
                "combined_3_lt_combined2": c3_beats_combined2,
                "combined_3_lt_both": c3_beats_both,
            })

        rho_mc, p_mc = spearmanr(marg, conf, nan_policy="omit")
        rho_mh, p_mh = spearmanr(marg, h, nan_policy="omit")
        spear_rows.append({
            "dataset": ds_key,
            "model": model,
            "n": n,
            "small_n": small_n,
            "spearman_margin_confidence": float(rho_mc) if rho_mc == rho_mc else np.nan,
            "p_margin_confidence": float(p_mc) if p_mc == p_mc else np.nan,
            "spearman_margin_entropy": float(rho_mh) if rho_mh == rho_mh else np.nan,
            "p_margin_entropy": float(p_mh) if p_mh == p_mh else np.nan,
            "margin_mean_std": float(np.std(marg, ddof=1)) if n > 1 else 0.0,
            "confidence_std": float(np.std(conf, ddof=1)) if n > 1 else 0.0,
            "entropy_std": float(np.std(h, ddof=1)) if n > 1 else 0.0,
            "mean_accuracy": float(np.mean(y)),
        })

    return pd.DataFrame(curve_rows), pd.DataFrame(aurc_tidy), pd.DataFrame(spear_rows)


ALL_CURVES, ALL_AURC, ALL_SPEAR = [], [], []
for ds_key, df in FRAMES.items():
    print(f"\n===== {DATASETS[ds_key]['label']} =====")
    curves, aurc, spear = run_dataset(ds_key, df)
    ALL_CURVES.append(curves)
    ALL_AURC.append(aurc)
    ALL_SPEAR.append(spear)
    wide = aurc.pivot(index=["model", "n", "small_n"], columns="signal", values="AURC").reset_index()
    print(wide.round(4).to_string(index=False))

df_curves = pd.concat(ALL_CURVES, ignore_index=True)
df_aurc = pd.concat(ALL_AURC, ignore_index=True)
df_spear = pd.concat(ALL_SPEAR, ignore_index=True)


In [ ]:
# === Write CSVs + headline ====================================================
aurc_path = OUT_DIR / "rq4_aurc_margin_benchmark.csv"
spear_path = OUT_DIR / "rq4_spearman_margin.csv"
(
    df_aurc.sort_values(["dataset", "model", "signal"])
    .to_csv(aurc_path, index=False)
)
df_spear.to_csv(spear_path, index=False)
print(f"Wrote {aurc_path} ({len(df_aurc):,} rows)")
print(f"Wrote {spear_path} ({len(df_spear):,} rows)")

for ds_key in DATASETS:
    sub = df_curves[(df_curves["dataset"] == ds_key) & (df_curves["signal"] != "margin_min")]
    p = OUT_DIR / f"rq4_risk_coverage_margin_{ds_key.lower()}.csv"
    (
        sub[["dataset", "model", "signal", "coverage", "selective_accuracy", "risk", "n", "small_n"]]
        .sort_values(["model", "signal", "coverage"], ascending=[True, True, False])
        .to_csv(p, index=False)
    )
    print(f"Wrote {p} ({len(sub):,} rows)")

print("\n=== Spearman (is margin a new axis?) ===")
print(df_spear.round(3).to_string(index=False))

# One row per (dataset, model) for the combined_3 test
cells = (
    df_aurc[df_aurc["signal"] == "combined_3"]
    [["dataset", "model", "n", "small_n", "AURC",
      "best_single_signal", "best_single_AURC",
      "combined_3_lt_best_single", "combined_3_lt_combined2", "combined_3_lt_both"]]
    .copy()
)
c2 = df_aurc[df_aurc["signal"] == "combined"][["dataset", "model", "AURC"]].rename(
    columns={"AURC": "AURC_combined2"}
)
cells = cells.merge(c2, on=["dataset", "model"])
n_cells = len(cells)
n_both = int(cells["combined_3_lt_both"].sum())
print(f"\n=== HEADLINE: combined_3 strictly below best single AND old combined ===")
print(f"  {n_both} / {n_cells} model-cells")
if n_both:
    print(cells.loc[cells["combined_3_lt_both"], ["dataset", "model", "n", "AURC", "best_single_signal", "best_single_AURC", "AURC_combined2"]].round(4).to_string(index=False))
else:
    print("  (none)")
print("\nPer cell:")
show = cells.merge(
    df_aurc[df_aurc["signal"].isin(["entropy", "confidence", "margin"])]
    .pivot(index=["dataset", "model"], columns="signal", values="AURC")
    .reset_index(),
    on=["dataset", "model"],
)
for _, r in show.iterrows():
    flag_n = f"  [n={int(r['n'])} SMALL]" if r["small_n"] else f"  n={int(r['n'])}"
    win = "  *** combined_3 beats both ***" if r["combined_3_lt_both"] else ""
    print(
        f"  {r['dataset']:12s} {r['model']:28s}{flag_n}\n"
        f"      ent={r['entropy']:.4f}  conf={r['confidence']:.4f}  marg={r['margin']:.4f}  "
        f"c2={r['AURC_combined2']:.4f}  c3={r['AURC']:.4f}  best={r['best_single_signal']}{win}"
    )


In [ ]:
# === Per-dataset risk–coverage panels =========================================
SIGNAL_LABELS = {
    "entropy":     "Semantic entropy",
    "confidence":  "Mapping confidence",
    "margin":      "UMLS margin",
    "random":      "Random (no ranking)",
    "combined_2":  "Entropy, then confidence if tied",
    "combined_3":  "Mean rank of entropy + confidence + margin",
}
_plot_specs = [
    ("entropy", "-", "#C44E52", SIGNAL_LABELS["entropy"]),
    ("confidence", "-.", "#4C72B0", SIGNAL_LABELS["confidence"]),
    ("margin", "-", "#E69F00", SIGNAL_LABELS["margin"]),
    ("random", "--", "#55A868", SIGNAL_LABELS["random"]),
    ("combined", "--", "#7a52a8", SIGNAL_LABELS["combined_2"]),
    ("combined_3", ":", "#882255", SIGNAL_LABELS["combined_3"]),
]

SHORT = {
    "Mistral-7B-Instruct-v0.1": "Mistral-7B",
    "Llama3-OpenBioLLM-8B": "OpenBioLLM-8B",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B",
}

for ds_key, cfg in DATASETS.items():
    sub_all = df_curves[(df_curves["dataset"] == ds_key) & (df_curves["signal"] != "margin_min")]
    models = [m for m in MODEL_ORDER if m in set(sub_all["model"])]
    n_models = len(models)
    ncols = min(4, n_models)
    nrows = int(np.ceil(n_models / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(4.0 * ncols, 3.5 * nrows),
        sharex=False, sharey=True,
    )
    axes = np.atleast_1d(axes).ravel()

    n_by_model = (
        sub_all.drop_duplicates(["model"])[["model", "n", "small_n"]]
        .set_index("model")
    )

    for ax, model in zip(axes, models):
        sub = sub_all[sub_all["model"] == model]
        for signal, style, color, label in _plot_specs:
            s = sub[sub["signal"] == signal].sort_values("coverage")
            if s.empty:
                continue
            ax.plot(
                s["coverage"], s["risk"],
                linestyle=style, marker="o", markersize=3, linewidth=1.6,
                color=color, label=label,
            )
        n_m = int(n_by_model.loc[model, "n"])
        small = bool(n_by_model.loc[model, "small_n"])
        title = SHORT.get(model, model)
        if small:
            title = f"{title} (n={n_m}, SMALL)"
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("Coverage")
        ax.set_ylabel("Risk")
        ax.set_xlim(0.08, 1.02)
        ax.grid(True, alpha=0.3)
        ax.tick_params(labelbottom=True)

    for ax in axes[n_models:]:
        ax.axis("off")

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles, labels, loc="upper center", ncol=3, frameon=False,
        bbox_to_anchor=(0.5, 1.04),
    )
    fig.suptitle(
        f"RQ4 risk-coverage on {ds_key} "
        f"({SIGNAL_LABELS['margin']} + {SIGNAL_LABELS['combined_3']})\n"
        "risk = 1 − selective accuracy; lower is better",
        y=1.10, fontsize=12,
    )
    fig.tight_layout()
    out_local = OUT_DIR / f"rq4_risk_coverage_margin_{ds_key.lower()}.png"
    out_all = ALL_FIG / f"RQ4_risk_coverage_margin_{ds_key.lower()}.png"
    fig.savefig(out_local, dpi=160, bbox_inches="tight")
    fig.savefig(out_all, dpi=160, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {out_local}")
    print(f"Saved {out_all}")


## Paired bootstrap CIs and combined_3 win tests

Concept lane only. Reuses `selective_curve` / `aurc_from_curve` and the existing ranking
functions (entropy inverted; confidence and margin high=safe; lexicographic 2-signal;
mean-rank `combined_3`). B = 2000, `np.random.default_rng(42)`. One resampled index set
per iteration, applied to every signal. MedMentions encoders appear in Part 1 CIs only.
Does not overwrite existing AURC/curve CSVs.


In [ ]:
# === Paired bootstrap CIs and combined_3 win tests (concept lane) ============
# Reuses selective_curve / aurc_from_curve / order_* / percentile_safe from c2.
# Does NOT reimplement AURC. Does NOT overwrite existing score CSVs.
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
B = 2000
BOOT_RNG = np.random.default_rng(SEED)
print(f"Bootstrap seed: np.random.default_rng({SEED}); B={B}")
print(
    "NOTE: best_single is chosen on the FULL sample, then tested on bootstraps "
    "of the same data — mildly optimistic. combined_3 vs confidence is reported "
    "separately to cover that."
)

SIGNALS_BOOT = ["entropy", "confidence", "margin", "combined", "combined_3"]
MM_ENCODERS = {"BERT-base", "BioBERT", "PubMedBERT"}
HEADLINE = [
    ("MedMentions", "FLAN-T5-base"),
    ("MedMentions", "BioMistral-7B"),
    ("MedMentions", "Mistral-7B-Instruct-v0.1"),
    ("MedMentions", "Llama3-OpenBioLLM-8B"),
    ("MedMentions", "Meta-Llama-3-8B-Instruct"),
    ("CADEC", "FLAN-T5-base"),
]
POINT_CSV = OUT_DIR / "rq4_aurc_margin_benchmark.csv"
assert POINT_CSV.is_file(), POINT_CSV
point_tbl = pd.read_csv(POINT_CSV)
point_n = (
    point_tbl.drop_duplicates(["dataset", "model"])[["dataset", "model", "n"]]
    .set_index(["dataset", "model"])["n"].to_dict()
)
point_aurc = {
    (r.dataset, r.model, r.signal): float(r.AURC)
    for r in point_tbl.itertuples(index=False)
}


def load_concept_cell_frames():
    """Same load as c3, but KEEP MedMentions encoders (needed for Part 1 CIs)."""
    frames = {}
    for ds_key, cfg in DATASETS.items():
        df = pd.read_csv(cfg["path"])
        ecol, acol, ccol = cfg["entropy_col"], cfg["accuracy_col"], cfg["confidence_col"]
        need = ["instance_id", "model_name", ecol, acol, ccol, "margin_mean"]
        miss = [c for c in need if c not in df.columns]
        assert not miss, f"{ds_key} missing {miss}"
        out = df[need].rename(columns={
            ecol: "entropy", acol: "accuracy", ccol: "confidence",
        }).copy()
        for col in ["entropy", "accuracy", "confidence", "margin_mean"]:
            out[col] = pd.to_numeric(out[col], errors="coerce")
        before = len(out)
        out = out.dropna(subset=["entropy", "accuracy", "confidence", "margin_mean"]).copy()
        out["entropy"] = out["entropy"].clip(lower=0.0)
        out["dataset"] = ds_key
        print(f"{ds_key}: kept {len(out):,}/{before:,} defined-margin rows")
        frames[ds_key] = out
    return frames


def aurc_of(y, order):
    """Existing AURC: trapezoid of selective_curve risk vs coverage."""
    curve = selective_curve(y, order)
    return aurc_from_curve(
        [r["coverage"] for r in curve],
        [r["risk"] for r in curve],
    )


def rank_orders(h, conf, marg):
    """Existing ranking directions; combiners as defined in c4."""
    h_safe = percentile_safe(h, higher_is_safer=False)
    c_safe = percentile_safe(conf, higher_is_safer=True)
    m_safe = percentile_safe(marg, higher_is_safer=True)
    combined_3 = (h_safe + c_safe + m_safe) / 3.0
    return {
        "entropy": order_entropy(h, conf),
        "confidence": order_confidence(h, conf),
        "margin": np.argsort(-marg, kind="mergesort"),
        "combined": order_combined(h, conf),
        "combined_3": np.argsort(-combined_3, kind="mergesort"),
    }


def holm_reject(pvals, alpha=0.05):
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    reject = np.zeros(m, dtype=bool)
    for rank, i in enumerate(order):
        if pvals[i] <= alpha / (m - rank):
            reject[i] = True
        else:
            break
    return reject


def boot_p(deltas):
    d = np.asarray(deltas, dtype=float)
    frac_ge = float(np.mean(d >= 0.0))
    frac_le = float(np.mean(d <= 0.0))
    return float(min(1.0, 2.0 * min(frac_ge, frac_le)))


FRAMES_BOOT = load_concept_cell_frames()

ci_rows = []
win_rows = []
cells = []
print("\n=== n per cell (must match figures / rq4_aurc_margin_benchmark.csv) ===")
n_ok = True
for ds_key, df in FRAMES_BOOT.items():
    models = [m for m in MODEL_ORDER if m in set(df.model_name)]
    for model in models:
        g = df[df["model_name"] == model].reset_index(drop=True)
        n = len(g)
        expected = point_n.get((ds_key, model))
        flag = ""
        if ds_key == SMALL_N_DATASET and model == SMALL_N_MODEL:
            flag = "  [SMALL-n FLAG]"
        match = (expected is not None) and (int(expected) == n)
        if not match:
            n_ok = False
        print(
            f"  {ds_key:12s} {model:28s} n={n}"
            f"  table_n={expected}  {'OK' if match else 'MISMATCH'}{flag}"
        )
        cells.append((ds_key, model, g, n))
assert n_ok, "n mismatch vs existing AURC table — abort"

print("\n=== Part 1+2: paired bootstrap (one index set per iteration, all signals) ===")
for ds_key, model, g, n in cells:
    y = g["accuracy"].to_numpy(dtype=float)
    h = g["entropy"].to_numpy(dtype=float)
    conf = g["confidence"].to_numpy(dtype=float)
    marg = g["margin_mean"].to_numpy(dtype=float)

    orders0 = rank_orders(h, conf, marg)
    point = {sig: aurc_of(y, orders0[sig]) for sig in SIGNALS_BOOT}
    for sig in SIGNALS_BOOT:
        stored = point_aurc.get((ds_key, model, sig))
        if stored is not None:
            delta_pt = abs(point[sig] - stored)
            if delta_pt > 1e-8:
                print(
                    f"  WARN point AURC drift {ds_key} {model} {sig}: "
                    f"recompute={point[sig]:.6f} table={stored:.6f} |d|={delta_pt:.2e}"
                )
            point[sig] = stored  # keep published table values unchanged

    singles = ["entropy", "confidence", "margin"]
    best_single = min(singles, key=lambda s: point[s])

    boot = {sig: np.empty(B, dtype=float) for sig in SIGNALS_BOOT}
    idx_all = np.arange(n)
    for b in range(B):
        idx = BOOT_RNG.choice(idx_all, size=n, replace=True)
        y_b = y[idx]
        orders_b = rank_orders(h[idx], conf[idx], marg[idx])
        for sig in SIGNALS_BOOT:
            boot[sig][b] = aurc_of(y_b, orders_b[sig])

    for sig in SIGNALS_BOOT:
        lo, hi = np.percentile(boot[sig], [2.5, 97.5])
        ci_rows.append({
            "dataset": ds_key,
            "model": model,
            "n": n,
            "signal": sig,
            "aurc_point": point[sig],
            "ci_low": float(lo),
            "ci_high": float(hi),
        })

    skip_win = ds_key == "MedMentions" and model in MM_ENCODERS
    if skip_win:
        print(f"  {ds_key:12s} {model:28s} Part 1 CIs only (MM encoder; not a win cell)")
        continue

    d_best = boot["combined_3"] - boot[best_single]
    d_conf = boot["combined_3"] - boot["confidence"]
    d_c2 = boot["combined_3"] - boot["combined"]
    d_best_pt = point["combined_3"] - point[best_single]
    d_conf_pt = point["combined_3"] - point["confidence"]
    d_c2_pt = point["combined_3"] - point["combined"]
    lo_b, hi_b = np.percentile(d_best, [2.5, 97.5])
    lo_c, hi_c = np.percentile(d_conf, [2.5, 97.5])
    lo_2, hi_2 = np.percentile(d_c2, [2.5, 97.5])
    p_best = boot_p(d_best)
    p_conf = boot_p(d_conf)
    p_c2 = boot_p(d_c2)
    sig_best = bool(hi_b < 0.0)
    win_rows.append({
        "dataset": ds_key,
        "model": model,
        "n": n,
        "best_single": best_single,
        "delta_point": d_best_pt,
        "delta_ci_low": float(lo_b),
        "delta_ci_high": float(hi_b),
        "p_value": p_best,
        "significant": sig_best,
        "holm_significant": False,  # filled below for headline 6
        "delta_vs_confidence": d_conf_pt,
        "delta_vs_confidence_ci_low": float(lo_c),
        "delta_vs_confidence_ci_high": float(hi_c),
        "p_vs_confidence": p_conf,
        "significant_vs_confidence": bool(hi_c < 0.0),
        "delta_vs_combined2": d_c2_pt,
        "delta_vs_combined2_ci_low": float(lo_2),
        "delta_vs_combined2_ci_high": float(hi_2),
        "p_vs_combined2": p_c2,
        "significant_vs_combined2": bool(hi_2 < 0.0),
    })
    print(
        f"  {ds_key:12s} {model:28s} n={n} best_single={best_single:10s} "
        f"d3-best={d_best_pt:+.4f} CI[{lo_b:+.4f},{hi_b:+.4f}] "
        f"p={p_best:.4g} sig={sig_best}"
    )

df_ci = pd.DataFrame(ci_rows)
df_win = pd.DataFrame(win_rows)

# Holm–Bonferroni across the 6 pre-specified headline tests only
df_win["headline"] = [
    (r.dataset, r.model) in HEADLINE for r in df_win.itertuples(index=False)
]
head = df_win[df_win["headline"]].copy()
assert len(head) == 6, f"expected 6 headline rows, got {len(head)}"
holm = holm_reject(head["p_value"].to_numpy(), alpha=0.05)
df_win.loc[head.index, "holm_significant"] = holm

print("\n=== Part 3 — headline 6 (point-estimate wins unchanged) ===")
print("cell                         delta     95% CI              p         sig  holm")
for r, hflag in zip(head.itertuples(index=False), holm):
    print(
        f"  {r.dataset:12s} {r.model:28s} {r.delta_point:+.4f}  "
        f"[{r.delta_ci_low:+.4f}, {r.delta_ci_high:+.4f}]  "
        f"{r.p_value:.4g}  {str(bool(r.significant)):3s}  {bool(hflag)}"
    )

n_sig = int(head["significant"].sum())
n_holm = int(np.sum(holm))
cross = head[~head["significant"]]
print(
    f"\ncombined_3 vs best-single: {n_sig} of 6 headline cells significant at 95% "
    f"({n_holm} survive Holm)."
)
if len(cross):
    print("Point-wins whose 95% CI crosses zero (NOT significant):")
    print(cross[["dataset", "model", "delta_point", "delta_ci_low", "delta_ci_high", "p_value"]].to_string(index=False))
else:
    print("No headline cell has a CI that crosses zero.")

ci_path = OUT_DIR / "rq4_aurc_bootstrap_ci.csv"
win_path = OUT_DIR / "rq4_combined3_wintest.csv"
assert ci_path.resolve() != POINT_CSV.resolve()
df_ci.to_csv(ci_path, index=False)
df_win.drop(columns=["headline"]).to_csv(win_path, index=False)
print("\nSaved", ci_path)
print("Saved", win_path)
print("Did not modify", POINT_CSV.name)
